# Final experiments, part 1: does label history explain the gap?

At the milestone, LightGBM with relational aggregates (0.734 test AUROC) beat the GNN without text (0.636). Its most important features were the **label history** of a trial's linked entities: how earlier trials with the same condition, sponsor, sites and interventions turned out. A 2-layer relational GNN never sees those labels, and with junction tables in between it never even reaches another trial.

This notebook tests that directly, on both tasks, 3 seeds each:

| Step | Run | Question |
|---|---|---|
| 1 | LightGBM aggregates **without** label history | Does removing history bring LightGBM down to GNN level? |
| 2 | GNN, no text, 30 epochs | Fair baseline (the milestone run had only 10 epochs) |
| 3 | GNN, no text **+ label history** | Does giving the GNN the history close the gap? |
| 4 | GNN with GloVe text, 30 epochs | RelBench's reference setup |
| 5 | GNN with text **+ label history** | Do the two effects add up? |

**Runtime:** L4 GPU with High-RAM, about 3–4 hours in total. With Colab Pro+ background execution you can close the tab. Results go to `results_final/` on Drive.

In [ ]:
REPO_URL = "https://github.com/mbermad/relational-trials.git"
import os
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/cs224w-relational-trials"
MILESTONE = f"{WORK}/results"          # milestone results (LightGBM table / agg reused)
RESULTS = f"{WORK}/results_final"      # everything from now on
CACHE = f"{WORK}/cache"
os.makedirs(RESULTS, exist_ok=True)
print("Results ->", RESULTS)

In [ ]:
%cd /content
!rm -rf relational-trials && git clone -q {REPO_URL}
%cd /content/relational-trials
!git log --oneline -1
!pip install -q -e . torch_geometric pytorch_frame sentence-transformers

In [ ]:
import torch
TORCH = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"  # pyg-lib wheels are built per minor version
CUDA = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
print("torch", TORCH, "| CUDA", CUDA)
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
import pyg_lib, torch_geometric, relbench, psutil
print("pyg-lib", pyg_lib.__version__, "| PyG", torch_geometric.__version__, "| relbench", relbench.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print(f"RAM: {psutil.virtual_memory().total / 2**30:.0f} GB (need about 40+ GB for the text GNN)")

Tests first (about 1 minute): every model, including the new label-history GNN, trains on a tiny synthetic copy of the schema. **Expect 19 passed.** If anything fails, stop and send the error.

In [ ]:
!pytest -q

Reuse the milestone LightGBM runs (same code, same seeds) so they appear in the final table.

In [ ]:
import shutil, glob
for d in glob.glob(f"{MILESTONE}/runs/*/lgbm_*") + glob.glob(f"{MILESTONE}/preds/*/lgbm_*"):
    dst = d.replace(MILESTONE, RESULTS)
    if not os.path.exists(dst):
        shutil.copytree(d, dst)
print(sorted(glob.glob(f"{RESULTS}/runs/*/*")))

## Step 1: LightGBM without label history (CPU, about 30 minutes)

In [ ]:
!python scripts/run.py lgbm --features agg_nohist --seeds 0 1 2 --n_trials 20 --results {RESULTS}

## Step 2: GNN without text, 30 epochs (about 25 minutes)

In [ ]:
!python scripts/run.py rdl --text none --epochs 30 --seeds 0 1 2 --cache_dir {CACHE} --results {RESULTS}

## Step 3: GNN without text + label history (about 25 minutes)
The trial's embedding gets the same time-safe history features LightGBM uses (a label counts only once it is known), encoded by a small MLP before the prediction head.

In [ ]:
!python scripts/run.py rdl --text none --history --epochs 30 --seeds 0 1 2 --cache_dir {CACHE} --results {RESULTS}

## Step 4: GNN with GloVe text, 30 epochs (about 1–1.5 hours; text embedding runs once per task)
Also saves the trial embeddings for the C3 hybrid later.

In [ ]:
!python scripts/run.py rdl --text glove --epochs 30 --seeds 0 1 2 --save_embeddings --cache_dir {CACHE} --results {RESULTS}

## Step 5: GNN with text + label history (about 40 minutes; reuses the cached text embeddings)

In [ ]:
!python scripts/run.py rdl --text glove --history --epochs 30 --seeds 0 1 2 --cache_dir {CACHE} --results {RESULTS}

## Results table and download

In [ ]:
!python scripts/run.py tables --results {RESULTS}
from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS}/tables/main.md").read()))

In [ ]:
shutil.make_archive("/content/final_results_part1", "zip", RESULTS, ".")
from google.colab import files
files.download("/content/final_results_part1.zip")